## Scikit-Learn Pipeline
This notebook explains how to build reproducible ML workflows using scikit-learn Pipelines.
You will learn leakage-safe preprocessing, cross-validation, tuning, and mixed-feature preprocessing.

### Setup and Imports
We import all required tools first so each next section runs independently and clearly.

In [ ]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pickle

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Libraries imported successfully')

The output confirms your environment is ready.
A common mistake is importing tools inside many cells and losing track of dependencies.
Quick tip: keep one clean setup cell at the top for reproducibility.

### The Problem: Manual Preprocessing Can Leak Test Information
If preprocessing is done manually, it is easy to accidentally fit transformers on test data.
That leaks future information and gives over-optimistic results.

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler_bad = StandardScaler()
scaler_bad.fit(X_test)  # Wrong: fitting on test data
X_train_bad = scaler_bad.transform(X_train)
X_test_bad = scaler_bad.transform(X_test)

print(f'Dataset shape: {X.shape}')
print('Manual preprocessing (wrong): scaler was fit on test data')
print(f'Train mean after wrong scaling: {X_train_bad.mean():.4f}')
print(f'Test mean after wrong scaling: {X_test_bad.mean():.4f}')

This demonstrates the leakage risk in manual workflows.
A common mistake is assuming any scaler fit is fine as long as transform works.
Quick tip: never call `fit` on test data for preprocessing steps.

### Solution: Simple Pipeline
A Pipeline chains preprocessing and model training in a fixed order.
During `fit`, transformers learn only from training data.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print('Pipeline steps:', list(pipeline.named_steps.keys()))
print(f'Pipeline accuracy: {acc:.4f}')
print('Scaler means (first 3 features):', pipeline.named_steps['scaler'].mean_[:3])
print('Model coef shape:', pipeline.named_steps['model'].coef_.shape)

The pipeline applies scaling first, then trains logistic regression safely.
A common mistake is manually transforming new data with a different scaler instance.
Quick tip: always use `pipeline.predict()` so preprocessing is automatically consistent.

### Cross-Validation with Pipeline
In each fold, the transformer is fit only on that fold's training subset.
This gives a reliable estimate without leakage.

In [ ]:
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
print('5-fold CV scores:', [f'{s:.4f}' for s in cv_scores])
print(f'Mean CV score: {cv_scores.mean():.4f}')
print(f'Std CV score: {cv_scores.std():.4f}')

These scores show how stable your model is across different folds.
A common mistake is doing cross-validation on pre-scaled full data.
Quick tip: pass the full pipeline into `cross_val_score`, not only the model.

### Hyperparameter Tuning Inside Pipeline
Use `GridSearchCV` with names like `step__parameter`.
This tunes model parameters while preserving the preprocessing flow.

In [ ]:
pipeline_tunable = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

param_grid = {
    'model__C': [0.01, 0.1, 1.0, 10.0],
    'model__solver': ['lbfgs', 'liblinear']
}

grid = GridSearchCV(pipeline_tunable, param_grid=param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print(f'Best CV score: {grid.best_score_:.4f}')
print(f'Test accuracy: {grid.score(X_test, y_test):.4f}')

Grid search checks multiple combinations and returns the best pipeline.
A common mistake is using `model_C` instead of `model__C` in the grid keys.
Quick tip: call `pipeline.get_params().keys()` to discover valid parameter names.

### Feature Engineering and Selection in a Pipeline
You can add polynomial expansion or automatic feature selection as steps.
This keeps transformations reproducible across training and inference.

In [ ]:
pipeline_poly = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', LogisticRegression(max_iter=2000, random_state=42))
])

pipeline_select = Pipeline([
    ('scaler', StandardScaler()),
    ('select', SelectKBest(score_func=f_classif, k=2)),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline_poly.fit(X_train, y_train)
pipeline_select.fit(X_train, y_train)

poly_acc = pipeline_poly.score(X_test, y_test)
select_acc = pipeline_select.score(X_test, y_test)

print(f'Original feature count: {X_train.shape[1]}')
print('Polynomial output feature count:', pipeline_poly.named_steps['poly'].n_output_features_)
print(f'Polynomial pipeline accuracy: {poly_acc:.4f}')
print(f'SelectKBest pipeline accuracy: {select_acc:.4f}')

Feature engineering can improve signal, while selection can reduce noise.
A common mistake is adding too many polynomial terms and causing overfitting.
Quick tip: compare train vs test scores when feature space grows.

### ColumnTransformer for Mixed Data
Real datasets often contain numeric and categorical features together.
`ColumnTransformer` lets us apply different preprocessing by column type.

In [ ]:
df = pd.DataFrame({
    'age': [25, 45, 35, 55, 30, 40, 29, 51],
    'income': [30000, 120000, 45000, 180000, 55000, 80000, 48000, 150000],
    'city': ['NYC', 'LA', 'NYC', 'NYC', 'LA', 'SF', 'SF', 'LA']
})
y_mixed = np.array([0, 1, 0, 1, 0, 1, 0, 1])

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), ['age', 'income']),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), ['city'])
])

mixed_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

mixed_pipeline.fit(df, y_mixed)
mixed_acc = mixed_pipeline.score(df, y_mixed)

print(df)
print(f'Mixed-feature pipeline accuracy: {mixed_acc:.4f}')

The pipeline applies scaling to numeric columns and one-hot encoding to categorical columns.
A common mistake is applying a numeric scaler directly to string columns.
Quick tip: keep preprocessing logic in `ColumnTransformer` so deployment matches training.

### Alternative Pipeline with Random Forest
Tree-based models usually do not require feature scaling.
But they can still be combined with feature selection in a pipeline.

In [ ]:
rf_pipeline = Pipeline([
    ('select', SelectKBest(score_func=f_classif, k=3)),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
rf_acc = rf_pipeline.score(X_test, y_test)
print(f'Random Forest pipeline accuracy: {rf_acc:.4f}')
print('Note: scaling is generally not required for trees')

This section shows that pipelines are useful beyond linear models.
A common mistake is adding scaling before trees without any reason.
Quick tip: choose preprocessing based on algorithm behavior, not habit.

### Save and Load a Trained Pipeline
Saving the whole pipeline stores both preprocessing and model together.
This is important for reliable deployment.

In [ ]:
save_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])
save_pipeline.fit(X_train, y_train)

save_path = Path('15_ml_workflow') / 'pipeline.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(save_pipeline, f)

with open(save_path, 'rb') as f:
    loaded_pipeline = pickle.load(f)

loaded_acc = loaded_pipeline.score(X_test, y_test)
print(f'Pipeline saved to: {save_path}')
print(f'Loaded pipeline accuracy: {loaded_acc:.4f}')

The loaded pipeline should reproduce the same preprocessing and prediction flow.
A common mistake is saving only the model and forgetting to save preprocessing.
Quick tip: always serialize the full pipeline for production use.

## Key Takeaways
- Pipelines prevent data leakage by controlling fit/transform order.
- Use `cross_val_score` and `GridSearchCV` directly with pipelines.
- Use `ColumnTransformer` for mixed numeric and categorical features.
- Trees usually do not need scaling, but still fit well in pipeline workflows.
- Save the full pipeline, not only the model, for consistent deployment.